<a href="https://colab.research.google.com/github/mariembohli27-sketch/gitgithub/blob/main/predictive-maintenance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
"""
=============================================================
 Predictive Maintenance System — NASA Turbofan Dataset
 Full ML Pipeline: RUL Regression + Failure Classification
=============================================================
"""
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib, os, joblib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report)

sns.set_theme(style="darkgrid", palette="muted")
RANDOM_STATE = 42; np.random.seed(RANDOM_STATE)
OUTPUT_DIR = "/home/claude/outputs"; os.makedirs(OUTPUT_DIR, exist_ok=True)
print("All dependencies loaded.")

# ── 1. DATA GENERATION ────────────────────────────────────────────────────────
print("\n" + "="*60 + "\nSTEP 1 — Data Generation (NASA-style synthetic)\n" + "="*60)

def generate_dataset(n_engines=150, seed=42):
    rng = np.random.default_rng(seed)
    records = []
    baselines = [518.67,641.82,1589.70,1400.60,14.62,21.61,554.36,2388.06,
                 9046.19,1.30,47.47,521.66,2388.02,8138.62,8.42,0.03,392.0,
                 2388.0,100.0,38.8,23.4]
    noise_lvl = [0.5,0.5,3.0,3.0,0.02,0.01,0.5,3.0,10.0,0.001,
                 0.2,0.5,3.0,10.0,0.01,0.001,0.5,3.0,0.2,0.1,0.1]
    deg_rates = [0,0,0.06,0.05,-0.0004,-0.0003,0,0,0,0,
                 -0.003,0,0,0,0,0,0.002,0,0,-0.001,-0.001]
    for eid in range(1, n_engines+1):
        max_c = rng.integers(150, 380)
        for t in range(1, max_c+1):
            op1 = rng.choice([-0.0007,-0.0004,0.0001], p=[0.4,0.3,0.3])
            op2 = rng.choice([0.0,0.00004,-0.0001],    p=[0.5,0.3,0.2])
            op3 = rng.choice([100.0,84.0,60.0],         p=[0.5,0.3,0.2])
            sensors = [b + dr*t*b*0.001 + rng.normal(0, max(n,0.001))
                       for b,n,dr in zip(baselines,noise_lvl,deg_rates)]
            records.append([eid,t,op1,op2,op3]+sensors)
    cols = (["engine_id","time_in_cycles","op_setting_1","op_setting_2","op_setting_3"] +
            [f"sensor_{i:02d}" for i in range(1,22)])
    return pd.DataFrame(records, columns=cols)

df_raw = generate_dataset()
sensor_cols = [c for c in df_raw.columns if c.startswith("sensor_")]
op_cols     = [c for c in df_raw.columns if c.startswith("op_")]
print(f"Shape: {df_raw.shape}  |  Engines: {df_raw['engine_id'].nunique()}  |  Sensors: {len(sensor_cols)}")
print(df_raw.head(3).to_string())

# ── 2. PREPROCESSING ──────────────────────────────────────────────────────────
print("\n" + "="*60 + "\nSTEP 2 — Preprocessing\n" + "="*60)
df = df_raw.copy()
low_var = df[sensor_cols].std()
useless = low_var[low_var < 0.01].index.tolist()
if useless: df.drop(columns=useless, inplace=True); sensor_cols=[c for c in sensor_cols if c not in useless]
print(f"Active sensors: {len(sensor_cols)}")
scaler = StandardScaler()
df[sensor_cols] = scaler.fit_transform(df[sensor_cols])
engine_ids = df["engine_id"].unique()
rng2 = np.random.default_rng(RANDOM_STATE); rng2.shuffle(engine_ids)
sp = int(len(engine_ids)*0.8)
train_df = df[df["engine_id"].isin(engine_ids[:sp])].copy()
test_df  = df[df["engine_id"].isin(engine_ids[sp:])].copy()
print(f"Train: {len(engine_ids[:sp])} engines ({len(train_df)} rows) | Test: {len(engine_ids[sp:])} engines ({len(test_df)} rows)")

# ── 3. TARGET ENGINEERING ─────────────────────────────────────────────────────
print("\n" + "="*60 + "\nSTEP 3 — Target Engineering\n" + "="*60)

def add_rul(d):
    mx = d.groupby("engine_id")["time_in_cycles"].max().rename("mx")
    d = d.join(mx, on="engine_id")
    d["RUL"] = d["mx"] - d["time_in_cycles"]
    return d.drop(columns=["mx"])

RUL_CAP = 125
train_df = add_rul(train_df); test_df = add_rul(test_df)
train_df["RUL_capped"] = train_df["RUL"].clip(upper=RUL_CAP)
test_df["RUL_capped"]  = test_df["RUL"].clip(upper=RUL_CAP)

n = len(sensor_cols)
t_s = sensor_cols[:n//3]; v_s = sensor_cols[n//3:2*n//3]; p_s = sensor_cols[2*n//3:]

def failure_type(row):
    t = row[t_s].mean(); v = row[v_s].mean(); p = row[p_s].mean()
    lf = 0.4 if row["RUL"] < 30 else 0.0
    if   t > 0.8 - lf:  return "Engine Failure"
    elif v > 0.7 - lf:  return "Mechanical Fault"
    elif p < -0.7 + lf: return "System Failure"
    else:                return "Normal Operation"

train_df["failure_type"] = train_df.apply(failure_type, axis=1)
test_df["failure_type"]  = test_df.apply(failure_type, axis=1)
le = LabelEncoder()
train_df["failure_label"] = le.fit_transform(train_df["failure_type"])
test_df["failure_label"]  = le.transform(test_df["failure_type"])
print("RUL range:", train_df["RUL"].min(), "–", train_df["RUL"].max())
print(train_df["failure_type"].value_counts().to_string())

# ── 4. EDA ────────────────────────────────────────────────────────────────────
print("\n" + "="*60 + "\nSTEP 4 — EDA\n" + "="*60)
fig = plt.figure(figsize=(22,16))
fig.suptitle("Predictive Maintenance — EDA Dashboard (NASA Turbofan)", fontsize=16, fontweight="bold")
gs = gridspec.GridSpec(3,3, hspace=0.50, wspace=0.35)

ax = fig.add_subplot(gs[0,0])
train_df["RUL"].hist(bins=40, ax=ax, color="#4C72B0", edgecolor="white")
ax.set_title("RUL Distribution",fontweight="bold"); ax.set_xlabel("RUL (cycles)"); ax.set_ylabel("Count")

ax = fig.add_subplot(gs[0,1])
ft = train_df["failure_type"].value_counts()
ax.bar(ft.index, ft.values, color=["#DD8452","#4C72B0","#55A868","#C44E52"][:len(ft)], edgecolor="white")
ax.set_title("Failure Type Distribution",fontweight="bold"); ax.tick_params(axis="x",rotation=20)

ax = fig.add_subplot(gs[0,2])
lifetimes = train_df.groupby("engine_id")["time_in_cycles"].max().sort_values()
ax.plot(range(len(lifetimes)),lifetimes.values,color="#55A868",lw=1.5)
ax.fill_between(range(len(lifetimes)),lifetimes.values,alpha=0.25,color="#55A868")
ax.set_title("Engine Lifetimes",fontweight="bold"); ax.set_xlabel("Engine (sorted)")

ax = fig.add_subplot(gs[1,:2])
corr_rul = train_df[sensor_cols+["RUL"]].corr()["RUL"].drop("RUL").sort_values()
top = pd.concat([corr_rul.head(5),corr_rul.tail(5)])
ax.barh(top.index,top.values,color=["#C44E52" if v<0 else "#4C72B0" for v in top.values],edgecolor="white")
ax.axvline(0,color="black",lw=0.8,ls="--"); ax.set_title("Sensor–RUL Correlations",fontweight="bold")

ax = fig.add_subplot(gs[1,2])
sns.heatmap(train_df[sensor_cols[:10]].corr(),ax=ax,cmap="coolwarm",center=0,linewidths=0.4,annot=False)
ax.set_title("Sensor Correlation (10)",fontweight="bold"); ax.tick_params(labelsize=7,rotation=45)

ax = fig.add_subplot(gs[2,:])
pal = sns.color_palette("tab10", 8)
for i,eid in enumerate(engine_ids[:8]):
    eng = train_df[train_df["engine_id"]==eid].sort_values("time_in_cycles")
    ax.plot(eng["time_in_cycles"],eng["RUL"],lw=1.8,alpha=0.85,color=pal[i],label=f"Eng {eid}")
ax.set_title("RUL Degradation Curves",fontweight="bold"); ax.set_xlabel("Cycle"); ax.set_ylabel("RUL"); ax.legend(ncol=2,fontsize=8)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/eda_dashboard.png", dpi=150, bbox_inches="tight"); plt.close()
print(f"EDA dashboard saved.")

# ── 5. MODELS ─────────────────────────────────────────────────────────────────
print("\n" + "="*60 + "\nSTEP 5 — Model Training\n" + "="*60)
FEATURES = [c for c in sensor_cols+op_cols if c in train_df.columns]
Xtr,Xte = train_df[FEATURES], test_df[FEATURES]
yR_tr,yR_te = train_df["RUL_capped"], test_df["RUL_capped"]
yC_tr,yC_te = train_df["failure_label"], test_df["failure_label"]

print("\nRegression:")
reg_models = {"Linear Regression": LinearRegression(),
              "Random Forest":     RandomForestRegressor(n_estimators=150,random_state=RANDOM_STATE,n_jobs=-1),
              "Gradient Boosting": GradientBoostingRegressor(n_estimators=150,learning_rate=0.08,max_depth=4,random_state=RANDOM_STATE)}
reg_res, reg_preds = {}, {}
for nm,m in reg_models.items():
    m.fit(Xtr,yR_tr); p=np.clip(m.predict(Xte),0,None)
    reg_res[nm]={"MAE":mean_absolute_error(yR_te,p),"RMSE":np.sqrt(mean_squared_error(yR_te,p)),"R2":r2_score(yR_te,p)}
    reg_preds[nm]=p
    print(f"  {nm:<28} MAE={reg_res[nm]['MAE']:.2f}  RMSE={reg_res[nm]['RMSE']:.2f}  R²={reg_res[nm]['R2']:.4f}")
best_reg = max(reg_res,key=lambda k:reg_res[k]["R2"])
print(f"Best: {best_reg}  R²={reg_res[best_reg]['R2']:.4f}")

print("\nClassification:")
cls_models = {"Logistic Regression": LogisticRegression(max_iter=1000,random_state=RANDOM_STATE),
              "Random Forest":       RandomForestClassifier(n_estimators=150,random_state=RANDOM_STATE,n_jobs=-1)}
cls_res, cls_preds = {}, {}
for nm,m in cls_models.items():
    m.fit(Xtr,yC_tr); p=m.predict(Xte)
    cls_res[nm]={"Accuracy":accuracy_score(yC_te,p),"F1":f1_score(yC_te,p,average="weighted",zero_division=0),
                 "Precision":precision_score(yC_te,p,average="weighted",zero_division=0),
                 "Recall":recall_score(yC_te,p,average="weighted",zero_division=0)}
    cls_preds[nm]=p
    print(f"  {nm:<28} Acc={cls_res[nm]['Accuracy']:.4f}  F1={cls_res[nm]['F1']:.4f}")
best_cls = max(cls_res,key=lambda k:cls_res[k]["F1"])
print(f"Best: {best_cls}  F1={cls_res[best_cls]['F1']:.4f}")

# ── 6. EVALUATION ─────────────────────────────────────────────────────────────
print("\n" + "="*60 + "\nSTEP 6 — Evaluation\n" + "="*60)
fig2,axes = plt.subplots(2,3,figsize=(22,13))
fig2.suptitle("Model Evaluation Dashboard",fontsize=16,fontweight="bold")

ax=axes[0,0]; bp=reg_preds[best_reg]
ax.scatter(yR_te,bp,alpha=0.25,s=8,color="#4C72B0",rasterized=True)
lm=[0,max(yR_te.max(),bp.max())]; ax.plot(lm,lm,"r--",lw=1.5,label="Ideal")
ax.set_title(f"Predicted vs Actual RUL\n{best_reg}",fontweight="bold"); ax.set_xlabel("Actual"); ax.set_ylabel("Predicted"); ax.legend()

ax=axes[0,1]; res=yR_te-bp
ax.hist(res,bins=50,color="#DD8452",edgecolor="white",alpha=0.85)
ax.axvline(0,color="red",ls="--",lw=1.5); ax.set_title("Residuals Distribution",fontweight="bold")

ax=axes[0,2]
nms_r=list(reg_res.keys()); r2s=[reg_res[m]["R2"] for m in nms_r]
cols_r=["#55A868" if m==best_reg else "#4C72B0" for m in nms_r]
bars=ax.bar(nms_r,r2s,color=cols_r,edgecolor="white")
ax.set_title("Regressor R² Comparison",fontweight="bold"); ax.set_ylim(0,1.05); ax.tick_params(axis="x",rotation=15)
for b,v in zip(bars,r2s): ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.01,f"{v:.3f}",ha="center",fontsize=9,fontweight="bold")

ax=axes[1,0]; cm=confusion_matrix(yC_te,cls_preds[best_cls])
sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",ax=ax,xticklabels=le.classes_,yticklabels=le.classes_,linewidths=0.5,cbar=False)
ax.set_title(f"Confusion Matrix\n{best_cls}",fontweight="bold"); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.tick_params(axis="x",rotation=20,labelsize=8); ax.tick_params(axis="y",labelsize=8)

ax=axes[1,1]
nms_c=list(cls_res.keys()); f1s=[cls_res[m]["F1"] for m in nms_c]
cols_c=["#55A868" if m==best_cls else "#DD8452" for m in nms_c]
bars2=ax.bar(nms_c,f1s,color=cols_c,edgecolor="white")
ax.set_title("Classifier F1 Comparison",fontweight="bold"); ax.set_ylim(0,1.1)
for b,v in zip(bars2,f1s): ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.01,f"{v:.3f}",ha="center",fontsize=9,fontweight="bold")

ax=axes[1,2]; bm=reg_models[best_reg]
if hasattr(bm,"feature_importances_"):
    imp=pd.Series(bm.feature_importances_,index=FEATURES).sort_values()
    imp.tail(14).plot(kind="barh",ax=ax,color="#4C72B0",edgecolor="white")
    ax.set_title("Feature Importance (RUL)",fontweight="bold")
else:
    pd.Series(np.abs(bm.coef_),index=FEATURES).sort_values().tail(14).plot(kind="barh",ax=ax,color="#4C72B0",edgecolor="white")
    ax.set_title("Feature |Coefficients|",fontweight="bold")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/evaluation_dashboard.png",dpi=150,bbox_inches="tight"); plt.close()
print("Evaluation dashboard saved.")
print(f"\nClassification Report ({best_cls}):")
print(classification_report(yC_te,cls_preds[best_cls],target_names=le.classes_,zero_division=0))

# ── 7. PREDICTION SYSTEM ──────────────────────────────────────────────────────
print("="*60 + "\nSTEP 7 — Prediction System\n" + "="*60)
best_reg_m = reg_models[best_reg]; best_cls_m = cls_models[best_cls]

def predict_maintenance(reading):
    row = pd.DataFrame([reading])
    for col in FEATURES:
        if col not in row: row[col]=0.0
    row = row[FEATURES]
    rul  = float(np.clip(best_reg_m.predict(row)[0],0,None))
    cls  = best_cls_m.predict(row)[0]
    prob = best_cls_m.predict_proba(row)[0] if hasattr(best_cls_m,"predict_proba") else None
    ftype= le.inverse_transform([cls])[0]
    risk = max(0,min(100,int((1-rul/RUL_CAP)*100)))
    if rul<15:   rec="CRITICAL  — Immediate maintenance!"
    elif rul<40: rec="HIGH RISK — Schedule within 24-48 hrs."
    elif rul<80: rec="MODERATE  — Plan this week."
    else:        rec="NORMAL    — No immediate action."
    return {"predicted_rul":round(rul,1),"failure_type":ftype,"risk_score":risk,"recommendation":rec,
            "class_proba":{le.classes_[i]:round(p,3) for i,p in enumerate(prob)} if prob is not None else None}

print("\nPrediction demos:\n")
for cond,subset in [("Near failure (RUL<20)",test_df[test_df["RUL"]<20]),
                    ("Mid-life (40<RUL<80)",test_df[test_df["RUL"].between(40,80)]),
                    ("Healthy (RUL>100)",  test_df[test_df["RUL"]>100])]:
    if len(subset)==0: continue
    row = subset.iloc[0]
    result = predict_maintenance({c:row[c] for c in FEATURES})
    print(f"  {cond}")
    print(f"    Actual RUL: {row['RUL']:.0f}  Predicted: {result['predicted_rul']}  Risk: {result['risk_score']}/100")
    print(f"    Type: {result['failure_type']}  → {result['recommendation']}\n")

# ── 8. SAVE ────────────────────────────────────────────────────────────────────
print("="*60 + "\nSTEP 8 — Save Models\n" + "="*60)
joblib.dump(best_reg_m, f"{OUTPUT_DIR}/rul_regressor.pkl")
joblib.dump(best_cls_m, f"{OUTPUT_DIR}/failure_classifier.pkl")
joblib.dump(scaler,      f"{OUTPUT_DIR}/feature_scaler.pkl")
joblib.dump(le,          f"{OUTPUT_DIR}/label_encoder.pkl")
joblib.dump(FEATURES,    f"{OUTPUT_DIR}/feature_list.pkl")
print(f"Models saved to {OUTPUT_DIR}/")

print("\n" + "─"*58 + "\n  FINAL SUMMARY\n" + "─"*58)
print("Regression:")
for nm,r in reg_res.items():
    print(f"  {nm:<28} MAE={r['MAE']:.2f}  RMSE={r['RMSE']:.2f}  R²={r['R2']:.4f}{'  <best' if nm==best_reg else ''}")
print("Classification:")
for nm,r in cls_res.items():
    print(f"  {nm:<28} Acc={r['Accuracy']:.4f}  F1={r['F1']:.4f}{'  <best' if nm==best_cls else ''}")
print("\nPipeline complete!")

All dependencies loaded.

STEP 1 — Data Generation (NASA-style synthetic)
Shape: (39478, 26)  |  Engines: 150  |  Sensors: 21
   engine_id  time_in_cycles  op_setting_1  op_setting_2  op_setting_3   sensor_01   sensor_02    sensor_03    sensor_04  sensor_05  sensor_06   sensor_07    sensor_08    sensor_09  sensor_10  sensor_11   sensor_12    sensor_13    sensor_14  sensor_15  sensor_16   sensor_17    sensor_18   sensor_19  sensor_20  sensor_21
0          1               1       -0.0004      -0.00010          84.0  517.694482  641.168910  1590.178903  1399.721302  14.619658  21.601463  554.799699  2390.393376  9046.850307   1.301127  47.563359  521.230354  2389.126252  8129.031174   8.428785   0.029950  391.908353  2385.957211  100.244508  38.784508  23.357144
1          1               2       -0.0007       0.00000         100.0  518.876366  642.035411  1596.315707  1399.520815  14.609743  21.601849  554.667990  2391.446917  9045.050525   1.299160  47.304819  521.985296  2390.249763  8

In [10]:
import pandas as pd
import sys
import os
import joblib
import numpy as np

# Define RUL_CAP as it was a global constant in the original notebook
RUL_CAP = 125

# Define load_artifacts function (adapted from the original notebook's intent)
def load_artifacts(output_dir="outputs"):
    """
    Loads pre-trained models, scaler, label encoder, and feature list.
    """
    artifacts = {}
    artifacts["reg_model"] = joblib.load(os.path.join(output_dir, "rul_regressor.pkl"))
    artifacts["cls_model"] = joblib.load(os.path.join(output_dir, "failure_classifier.pkl"))
    artifacts["scaler"] = joblib.load(os.path.join(output_dir, "feature_scaler.pkl"))
    artifacts["label_encoder"] = joblib.load(os.path.join(output_dir, "label_encoder.pkl"))
    artifacts["feature_cols"] = joblib.load(os.path.join(output_dir, "feature_list.pkl"))
    return artifacts

# Define predict_maintenance function (adapted to take artifacts as arguments)
# This function is derived from the implementation in the first cell,
# adapted to use the loaded artifacts explicitly.
def predict_maintenance(reading, reg_model, cls_model, label_encoder, feature_cols, rul_cap_val=RUL_CAP):
    """
    Predicts RUL and failure type for a given engine reading using loaded artifacts.
    """
    row = pd.DataFrame([reading])
    for col in feature_cols:
        if col not in row:
            row[col] = 0.0 # Assign default value for missing features
    row = row[feature_cols]

    rul  = float(np.clip(reg_model.predict(row)[0], 0, None))
    cls  = cls_model.predict(row)[0]
    prob = cls_model.predict_proba(row)[0] if hasattr(cls_model, "predict_proba") else None
    ftype= label_encoder.inverse_transform([cls])[0]
    risk = max(0, min(100, int((1 - rul / rul_cap_val) * 100)))

    if rul < 15:
        rec = "CRITICAL  — Immediate maintenance!"
    elif rul < 40:
        rec = "HIGH RISK — Schedule within 24-48 hrs."
    elif rul < 80:
        rec = "MODERATE  — Plan this week."
    else:
        rec = "NORMAL    — No immediate action."
    return {"predicted_rul": round(rul, 1), "failure_type": ftype, "risk_score": risk, "recommendation": rec,
            "class_proba": {label_encoder.classes_[i]: round(p, 3) for i, p in enumerate(prob)} if prob is not None else None}

# The sys.path.append("./src") line is no longer strictly necessary if functions are defined directly
# But keeping it for context or if other src modules were intended.
sys.path.append("./src")  # add src folder to Python path

# Load all saved artifacts (regressor, classifier, scaler, etc.)
artifacts = load_artifacts(output_dir="outputs")

reg_model     = artifacts["reg_model"]
cls_model     = artifacts["cls_model"]
scaler        = artifacts["scaler"]
label_encoder = artifacts["label_encoder"]
feature_cols  = artifacts["feature_cols"]


FileNotFoundError: [Errno 2] No such file or directory: 'outputs/rul_regressor.pkl'

In [12]:
import os

# Affiche le dossier courant où Python cherche les fichiers
print(os.getcwd())
# Affiche tous les fichiers et dossiers dans le dossier courant
print(os.listdir())

/content
['.config', 'sample_data']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')